
# 20 — Data Processing (Cars 4 You)

**Scope:** Perform cleaning & preprocessing (no modeling).  
Outputs a clean dataset and a JSON processing report for traceability.

ONLY Rulebased Cleaning (no ML), Missing-Handling, Encoding-Preparation.  
No Fit, no Scalers, no Target.
We want to avoid any data leakage.

That is the reason, why we don't have to split the data into train/test here.

## 1. Configuration & Paths

In [7]:
import os, re, math, warnings
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np
import re


import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("mode.copy_on_write", True)
warnings.filterwarnings("ignore")

RANDOM_STATE = 42  # for reproducibility of any sampling


## 2. Data Access & Loading

In [8]:
# Load the data paths
data_dir = "../data/"

# Load the raw data into a pandas dataframe
train = pd.read_csv(os.path.join(data_dir, "train.csv"))
test = pd.read_csv(os.path.join(data_dir, "test.csv"))

print("Loaded shape:", train.shape)
display(train.head(3))


Loaded shape: (75973, 14)


,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.0,0.0
1,53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.0,0.0
2,6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.0,0.0


## 4. Type Casting

In [9]:

def to_int_series(s):
    return pd.to_numeric(s, errors="coerce").round().astype("Int64")
def to_float_series(s):
    return pd.to_numeric(s, errors="coerce").astype(float)

num_cols_suggest = ["price","mileage","engineSize","mpg","tax","year","previousOwners"]
for col in num_cols_suggest:
    if col in train.columns:
        if col in ["year", "previousOwners"]:
            train[col] = to_int_series(train[col])
        else:
            train[col] = to_float_series(train[col])

for c in train.select_dtypes(include="object").columns:
    train[c] = train[c].astype("string").str.strip().replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})


train.head(2)

,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,69512,VW,Golf,2016,22290.0,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4,0.0
1,53000,Toyota,Yaris,2019,13790.0,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1,0.0


In [10]:
# Types of the columns
train.dtypes

carID                      int64
Brand             string[python]
model             string[python]
year                       Int64
price                    float64
transmission      string[python]
mileage                  float64
fuelType          string[python]
tax                      float64
mpg                      float64
engineSize               float64
paintQuality%            float64
previousOwners             Int64
hasDamage                float64
dtype: object

## 5. Deduplication

In [11]:
id_cols = [c for c in ["carID","id","ID"] if c in train.columns]
if id_cols:
    dup_n = train.duplicated(subset=id_cols).sum()
    train = train.drop_duplicates(subset=id_cols, keep="first")
else:
    dup_n = train.duplicated().sum()
    train = train.drop_duplicates(keep="first")

print(f"Duplicates removed: {dup_n} | New shape: {train.shape}")


Duplicates removed: 0 | New shape: (75973, 14)


## 6. Category Normalization (fuelType, transmission)

In [20]:
# Load the JSON mapping
with open("../mapping/fueltype_mapping.json", "r", encoding="utf-8") as f:
    CANON = json.load(f)

# Define a normalization function
def norm_fueltype(fueltype):
    if pd.isna(fueltype):
        return fueltype
    fueltype = fueltype.strip().lower()
    fueltype = re.sub(r'[.,-_]', ' ', fueltype)
    fueltype = ' '.join(fueltype.split())  # Remove extra spaces
    return fueltype


# Apply normalization and canonical mapping
train['fuelType'] = train['fuelType'].apply(norm_fueltype).map(CANON)

# Check results
print(train["fuelType"].value_counts(dropna=False))

fuelType
Petrol    41181
Diesel    30885
Hybrid     2180
NaN        1566
Other       161
Name: count, dtype: int64


In [13]:
# # Define a normalization function for fuelType
# def clean_fuelType(x):
#     if pd.isna(x):
#         return np.nan
#     s = str(x).strip().lower()

#     # handle common patterns
#     if re.search(r"etr|asolin", s):
#         return "petrol"
#     elif re.search(r"ies", s):
#         return "diesel"
#     elif re.search(r"ybri", s):
#         return "hybrid"
#     elif re.search(r"lect|ev", s):
#         return "electric"
#     elif re.search(r"the", s):
#         return "other"
#     elif re.search(r"unk", s):
#         return "unknown"
#     else:
#         # if none matches, should be the same as original value
#         return x
    

# # print NaN count before
# print(f"NaN count before: {train['fuelType'].isna().sum()}")
    
# # Apply it to the 'fuelType' column
# if "fuelType" in train.columns:
#     train["fuelType"] = train["fuelType"].apply(clean_fuelType)

# # Check results
# print(f"NaN count after: {train['fuelType'].isna().sum()}")
# print(train["fuelType"].value_counts(dropna=False))


# 7. Category Mapping for Brand and Model 
   
   

- First Step: Mapping of Brandnames with the corresponding JSON
  

In [14]:
# Load the mapping
with open("../mapping/brandname_mapping.json", "r", encoding="utf-8") as f:
    brandNameMapping = json.load(f)

# Normalization
def norm_brand(brand):
    if pd.isna(brand):
        return pd.NA
    brand = brand.strip().lower()
    brand = re.sub(r"[.,-_]", " ", brand)
    brand = " ".join(brand.split())
    return brand

# Cleaning (safe)
def clean_brand(col: pd.Series) -> pd.Series:
    nb = col.apply(norm_brand)
    mapped = nb.map(brandNameMapping)
    # Keep original where no mapping exists
    cleaned = col.where(mapped.isna(), mapped)
    return cleaned

# Apply cleaning
if "Brand" in train.columns:
    train["Brand"] = clean_brand(train["Brand"])

# Check result
print("\nUnique Brand in 'Brand':")
print(train["Brand"].unique())



Unique Brand in 'Brand':
<StringArray>
['Volkswagen', 'Toyota', 'Audi', 'Ford', 'BMW', 'Škoda', 'Opel', 'Mercedes-Benz', 'Hyundai', <NA>, 'pe', 'or', 'ercede']
Length: 13, dtype: string


In [15]:
# Zeige mir das Model an welches genau q ist 
print("\nEntries for model 'q':\n", train[train['model'] == 'Q'])

# Anzahl der Einträge für das Model q
print("\nNumber of entries for model 'q':", train[train['model'] == 'Q'].shape[0])


Entries for model 'q':
        carID Brand model  year    price transmission  mileage fuelType    tax   mpg  engineSize  paintQuality%  previousOwners  hasDamage
385      701  Audi     Q  2019  20999.0       Manual   5000.0   Petrol  145.0  47.1         1.0           36.0               1        0.0
1603    4859  Audi     Q  2020  31990.0    Semi-Auto   2166.0   Diesel  145.0  47.1         2.0           93.0               1        0.0
8686    4812  Audi     Q  2017  18771.0       Manual  15888.0   Petrol  150.0  52.3         1.4           99.0               2        0.0
10273   4724  Audi     Q  2017  16295.0    Automatic  85780.0   Diesel  200.0  47.9         2.0           72.0               3        0.0
13803    532  Audi     Q  2020  40990.0    Semi-Auto   5000.0   Petrol  145.0  32.1         2.0           37.0               1        0.0
15015   1866  Audi     Q  2024  10950.0       Manual  65542.0   Diesel  145.0  54.3         2.0           80.0               6        0.0
15386    

- Mapping the modelNames with the corresponding JSON

In [16]:
# Load mapping JSON (kann regex-only sein; ALIASES darf auch leer sein) --> This Mapping was created with the help of ChatGPT and manual adjustments and checkings 
with open("../mapping/modelname_mapping.json", "r", encoding="utf-8") as f:
    MM = json.load(f)

ALIASES = MM.get("aliases", {})            # optional
REGEX_RULES = MM.get("regex_rules", [])    # deine Regex-Regeln

def norm_model(x):
    """Lowercase, keep [a-z0-9+], strip spaces/hyphens/underscores."""
    if pd.isna(x):
        return np.nan
    s = str(x).lower()
    s = re.sub(r'[\s\-_]+', '', s)        # drop spaces, hyphens, underscores
    s = re.sub(r'[^a-z0-9\+]', '', s)     # keep letters, digits, plus
    return s

def post_canon_model(s):
    if pd.isna(s): return s
    t = str(s)

    # fehlendes 's' in '-Clas'
    t = re.sub(r'\b([ACEGSV]) Clas\b', r'\1-Class', t)
    t = re.sub(r'\b(GL[ABCES]) Clas\b', r'\1-Class', t)

    # zwei-Buchstaben-Klassen: "CL Class" -> "CL-Class"
    t = re.sub(r'\b([A-Z]{2}) Class\b', r'\1-Class', t)

    # a-/b-/c-/e-/s-/v-/g- vor "-Class" großschreiben
    t = re.sub(r'^([a-z])\-Class$', lambda m: m.group(1).upper() + "-Class", t)

    # GLA/GLB/GLC/GLE/GLS großschreiben, falls gemischt
    t = re.sub(r'^gl([abcse])\-Class$', lambda m: "GL" + m.group(1).upper() + "-Class", t, flags=re.I)

    # i/ix-Modelle auf korrekte Schreibweise
    t = re.sub(r'^I(10|20|30|40|800)$', r'i\1', t)
    t = re.sub(r'^IX(1|20|35)$', r'ix\1', t)

    # Audi Sport- und Kleinbuchstabenfälle
    t = re.sub(r'^tt$', 'TT', t, flags=re.I)
    t = re.sub(r'^r8$', 'R8', t, flags=re.I)
    t = re.sub(r'^sq7$', 'SQ7', t, flags=re.I)

    return t

train["model"] = train["model"].apply(post_canon_model)


def apply_regex_rules(norm_key: str):
    """
    Wendet die Regeln sequenziell an und gibt (result, matched) zurück.
    matched=True, sobald eine Regel etwas verändert hat.
    """
    out = norm_key
    matched = False
    for rule in REGEX_RULES:
        pat = rule.get("pattern", "")
        rep = rule.get("replace", "")
        new = re.sub(pat, rep, out)
        if new != out:
            matched = True
        out = new
    return out, matched

def clean_model(series: pd.Series) -> pd.Series:
    original = series.astype("string")

    # 1) Normalisieren für Lookup/Regex
    normed = original.apply(norm_model)

    # 2) Alias-Mapping (nur ersetzen, wo ein Alias existiert)
    alias_mapped = normed.map(ALIASES) if ALIASES else pd.Series(pd.NA, index=series.index)
    out = original.where(alias_mapped.isna(), alias_mapped)

    # 3) Regex-Fallback nur dort, wo noch nichts geändert wurde (oder NA)
    need = out.isna() | (out == original)
    if need.any():
        def _rx_or_na(k):
            if isinstance(k, str):
                res, ok = apply_regex_rules(k)
                return res if ok else pd.NA  # nur ersetzen, wenn wirklich ein Regex-Match stattfand
            return pd.NA
        rx_res = normed[need].apply(_rx_or_na)
        out.loc[need] = out.loc[need].where(rx_res.isna(), rx_res)

    return out


# Apply
if "model" in train.columns:
    train["model"] = clean_model(train["model"])


# Checks
unique_models = train["model"].unique()
print("\nUnique Model in 'model':\n", unique_models)
print(f"\nNumber of unique models: {len(unique_models)}")



Unique Model in 'model':
 <StringArray>
[        'Golf',        'Yaris',           'Q2',       'Fiesta',     '2 Series',     '3 Series',           'A3',      'Octavia',       'Passat',
        'Focus',
 ...
         'Hilu',     'glc clas', 'Urban Cruise',  'Land Cruise',       'ARTEON',     'gl class',         'Vers',     'Terracan',         's-ma',
       'arteon']
Length: 304, dtype: string

Number of unique models: 304


In [17]:
unique_models = train["model"].dropna().unique().tolist()
print("\nUnique Model in 'model':\n", unique_models)
print(f"\nNumber of unique models: {len(unique_models)}")



Unique Model in 'model':
 ['Golf', 'Yaris', 'Q2', 'Fiesta', '2 Series', '3 Series', 'A3', 'Octavia', 'Passat', 'Focus', 'Insignia', 'A-Class', 'Q3', 'Fabia', 'Ka+', 'GLC', 'i30', 'C-Class', 'Polo', 'E-Class', 'Q5', 'Up', 'C-HR', 'Mokka X', 'Corsa', 'Astra', 'TT', '5 Series', 'Aygo', '4 Series', 'SLK', 'Viva', 'T-Roc', 'EcoSport', 'Tucson', 'EcoSpor', 'X-Class', 'CL-Class', 'ix20', 'i20', 'Rapid', 'A1', 'Auris', 'Sharan', 'Adam', 'X3', 'A8', 'GLS', 'B-MAX', 'A4', 'Kona', 'i10', 'Mokka', 'S-MAX', 'X2', 'Crossland X', 'Tiguan', 'A5', 'GLE', 'Zafira', 'Ioniq', 'A6', 'Mondeo', 'Yeti Outdoor', 'X1', 'POLO', 'Scala', 'S-Class', '1 Series', 'Kamiq', 'Kuga', 'Tourneo Connect', 'Q7', 'GLA', 'Arteon', 'polo', 'SL', 'Santa Fe', 'Grandland X', 'i800', 'RAV4', 'Touran', 'Citigo', 'Roomster', 'Prius', 'Corolla', 'B-Class', 'Q', 'Kodiaq', 'V-Class', 'Caddy Maxi Life', 'Superb', 'Getz', 'Combo Life', 'Beetle', 'Galaxy', 'M3', 'GTC', 'X4', 'Ka', 'ix35', 'Grand Tourneo Connect', 'Shara', 'M4', 'Tourneo 

In [18]:
# Show the entries with model "kadjar"
print("\nEntries for model 'kadjar':\n", train[train['model'] == 'Kadjar'])


Entries for model 'kadjar':
        carID Brand   model  year    price transmission  mileage fuelType    tax    mpg  engineSize  paintQuality%  previousOwners  hasDamage
4171   56952  Opel  Kadjar  2018  12895.0       Manual  16546.0   Petrol  145.0  42.20         1.3           91.0               4        0.0
40381  60724  Opel  Kadjar  2019  15300.0       Manual  13983.0   Petrol  145.0  42.16         1.3           41.0               2        NaN
58913  59914  Opel  Kadjar  2019  15100.0       Manual  15101.0   Petrol  145.0  42.16         1.3           46.0               0        0.0


### We skip the Canonicalization of Models for now. to be revisited later.


## 10. Save Processed Data

In [19]:
# save the processed dataframe to data/processed_data
PROCESSED_CSV = os.path.join(data_dir, "processed_data/20_processed_train_data.csv")
print("Saving processed file to:", PROCESSED_CSV)
train.to_csv(PROCESSED_CSV, index=False)
print("✅ Saved. Rows with any remaining NaNs:", int(train.isna().any(axis=1).sum()))



Saving processed file to: ../data/processed_data/20_processed_train_data.csv


OSError: Cannot save file into a non-existent directory: '..\data\processed_data'